In [1]:
# -----------------------------------------------------------
# CELL 1: Load API Key and Connect to Database
# os, dotenv  → load the API key securely from .env file
# sqlite3     → connect to the Chinook database
# openai      → official Python client for the OpenAI API
# -----------------------------------------------------------

import os
import sqlite3
from dotenv import load_dotenv
from openai import OpenAI

# Load API key from .env file
load_dotenv()
client = OpenAI()

# Connect to Chinook database
conn = sqlite3.connect('../data/Chinook_Sqlite.sqlite')
cursor = conn.cursor()

print('API key loaded and database connected successfully!')


API key loaded and database connected successfully!


In [2]:
# -----------------------------------------------------------
# CELL 2: Load Schema String
# We re-run the schema extraction logic here so this notebook
# is self-contained and does not depend on running another
# notebook first. The schema_string is passed into the LLM
# prompt so the model knows what tables and columns exist.
# -----------------------------------------------------------

# Extract all table names
cursor.execute("SELECT name FROM sqlite_master WHERE type='table'")
tables = [row[0] for row in cursor.fetchall()]

# Extract columns for each table
schema = {}
for table in tables:
    cursor.execute(f'PRAGMA table_info({table})')
    columns = cursor.fetchall()
    schema[table] = [(col[1], col[2]) for col in columns]

# Format schema as a clean string for the LLM prompt
def format_schema_for_prompt(schema):
    schema_str = ''
    for table, columns in schema.items():
        schema_str += f'Table: {table}\n'
        for col_name, col_type in columns:
            schema_str += f'  - {col_name} ({col_type})\n'
        schema_str += '\n'
    return schema_str

schema_string = format_schema_for_prompt(schema)
print('Schema loaded successfully!')
print(f'Total tables: {len(schema)}')


Schema loaded successfully!
Total tables: 11


In [3]:
# -----------------------------------------------------------
# CELL 3: Build the Prompt
# The prompt is the instruction we send to the LLM.
# It has 3 parts:
#   1. The database schema (what tables/columns exist)
#   2. Instructions (return only SQL, no explanation)
#   3. The user's question
# This is the core of prompt engineering for our project.
# -----------------------------------------------------------

def build_prompt(schema_string, user_question):
    prompt = f"""You are an expert SQL assistant.

You are given the following database schema:
{schema_string}

Write a valid SQLite SQL query to answer the following question:
{user_question}

Rules:
- Return ONLY the SQL query, nothing else
- Do not include any explanation or markdown
- Do not use ```sql or ``` in your response
- Make sure the query is valid SQLite syntax
"""
    return prompt

# Test the prompt with a sample question
test_question = "Which artist has the most albums?"
prompt = build_prompt(schema_string, test_question)
print('Prompt built successfully!')
print('---')
print(prompt)


Prompt built successfully!
---
You are an expert SQL assistant.

You are given the following database schema:
Table: Album
  - AlbumId (INTEGER)
  - Title (NVARCHAR(160))
  - ArtistId (INTEGER)

Table: Artist
  - ArtistId (INTEGER)
  - Name (NVARCHAR(120))

Table: Customer
  - CustomerId (INTEGER)
  - FirstName (NVARCHAR(40))
  - LastName (NVARCHAR(20))
  - Company (NVARCHAR(80))
  - Address (NVARCHAR(70))
  - City (NVARCHAR(40))
  - State (NVARCHAR(40))
  - Country (NVARCHAR(40))
  - PostalCode (NVARCHAR(10))
  - Phone (NVARCHAR(24))
  - Fax (NVARCHAR(24))
  - Email (NVARCHAR(60))
  - SupportRepId (INTEGER)

Table: Employee
  - EmployeeId (INTEGER)
  - LastName (NVARCHAR(20))
  - FirstName (NVARCHAR(20))
  - Title (NVARCHAR(30))
  - ReportsTo (INTEGER)
  - BirthDate (DATETIME)
  - HireDate (DATETIME)
  - Address (NVARCHAR(70))
  - City (NVARCHAR(40))
  - State (NVARCHAR(40))
  - Country (NVARCHAR(40))
  - PostalCode (NVARCHAR(10))
  - Phone (NVARCHAR(24))
  - Fax (NVARCHAR(24))
  - Em

In [4]:
# -----------------------------------------------------------
# CELL 4: Send Prompt to LLM
# We send the prompt to GPT-4o-mini via the OpenAI API.
# The model reads the schema + question and returns a SQL query.
# We extract just the text content from the response.
# -----------------------------------------------------------

def get_sql_from_llm(prompt):
    response = client.chat.completions.create(
        model='gpt-4o-mini',
        messages=[
            {'role': 'system', 'content': 'You are an expert SQL assistant. Return only valid SQL queries.'},
            {'role': 'user', 'content': prompt}
        ],
        temperature=0  # temperature=0 means deterministic output (less creative, more precise)
    )
    # Extract the SQL text from the response
    sql_query = response.choices[0].message.content.strip()
    return sql_query

# Test with our sample question
sql_query = get_sql_from_llm(prompt)
print('SQL generated by LLM:')
print(sql_query)


SQL generated by LLM:
SELECT Artist.Name, COUNT(Album.AlbumId) AS AlbumCount
FROM Artist
JOIN Album ON Artist.ArtistId = Album.ArtistId
GROUP BY Artist.ArtistId
ORDER BY AlbumCount DESC
LIMIT 1;


In [5]:
# -----------------------------------------------------------
# CELL 5: Run SQL on the Database
# We execute the SQL query the LLM generated on the Chinook
# database and fetch the results.
# If the SQL is invalid, this will raise an error — we will
# handle that in the self-correction loop later.
# -----------------------------------------------------------

def run_sql(sql_query):
    cursor.execute(sql_query)
    results = cursor.fetchall()
    return results

# Run the SQL and print results
results = run_sql(sql_query)
print('Query results:')
for row in results:
    print(row)


Query results:
('Iron Maiden', 21)


In [6]:
# -----------------------------------------------------------
# CELL 6: Full End-to-End Pipeline
# This function combines all steps into one clean flow:
# question → prompt → LLM → SQL → database → result
# This is the core of our Text-to-SQL application.
# -----------------------------------------------------------

def text_to_sql_pipeline(user_question):
    print(f'Question: {user_question}')
    print('---')

    # Step 1: Build the prompt
    prompt = build_prompt(schema_string, user_question)

    # Step 2: Get SQL from LLM
    sql_query = get_sql_from_llm(prompt)
    print(f'Generated SQL: {sql_query}')
    print('---')

    # Step 3: Run SQL on database
    results = run_sql(sql_query)
    print(f'Results: {results}')
    return results

# Test with multiple questions
text_to_sql_pipeline("Which artist has the most albums?")
print()
text_to_sql_pipeline("What are the top 5 genres by number of tracks?")
print()
text_to_sql_pipeline("How many customers are from the USA?")


Question: Which artist has the most albums?
---
Generated SQL: SELECT Artist.Name, COUNT(Album.AlbumId) AS AlbumCount
FROM Artist
JOIN Album ON Artist.ArtistId = Album.ArtistId
GROUP BY Artist.ArtistId
ORDER BY AlbumCount DESC
LIMIT 1;
---
Results: [('Iron Maiden', 21)]

Question: What are the top 5 genres by number of tracks?
---
Generated SQL: SELECT g.Name, COUNT(t.TrackId) AS TrackCount
FROM Genre g
JOIN Track t ON g.GenreId = t.GenreId
GROUP BY g.GenreId
ORDER BY TrackCount DESC
LIMIT 5;
---
Results: [('Rock', 1297), ('Latin', 579), ('Metal', 374), ('Alternative & Punk', 332), ('Jazz', 130)]

Question: How many customers are from the USA?
---
Generated SQL: SELECT COUNT(*) FROM Customer WHERE Country = 'USA';
---
Results: [(13,)]


[(13,)]

In [7]:
# -----------------------------------------------------------
# CELL 7: Build the Correction Prompt
# When SQL fails, we send a new prompt to the LLM containing:
#   1. The SQL that failed
#   2. The exact error message
#   3. The original question
#   4. The schema string so LLM knows correct table/column names
# The LLM uses this to understand what went wrong and fix it.
# -----------------------------------------------------------

def build_correction_prompt(user_question, failed_sql, error_message, schema_string):
    prompt = f"""The following SQL query failed when run on a SQLite database:

{failed_sql}

Error message:
{error_message}

Here is the correct database schema:
{schema_string}

Please fix the SQL query to correctly answer the following question:
{user_question}

Rules:
- Return ONLY the corrected SQL query, nothing else
- Do not include any explanation or markdown
- Do not use ```sql or ``` in your response
- Make sure the query is valid SQLite syntax
- Only use table and column names that exist in the schema above
"""
    return prompt

print('Correction prompt function defined successfully!')

Correction prompt function defined successfully!


In [8]:
# -----------------------------------------------------------
# CELL 8: Self-Correction Loop
# This function wraps run_sql() with retry logic:
#   - Try running the SQL
#   - If it fails, send the error back to the LLM
#   - Ask the LLM to fix and retry
#   - Limit to max_retries attempts to avoid infinite loops
# -----------------------------------------------------------

def run_sql_with_correction(user_question, sql_query, max_retries=3):
    attempt = 0

    while attempt < max_retries:
        try:
            # Try running the SQL
            cursor.execute(sql_query)
            results = cursor.fetchall()
            if attempt > 0:
                print(f'SQL succeeded on attempt {attempt + 1}')
            return results  # success — return results

        except Exception as error:
            attempt += 1
            print(f'Attempt {attempt} failed: {error}')

            if attempt >= max_retries:
                print('Max retries reached. Could not fix the SQL.')
                return None

            # Build correction prompt with schema and ask LLM to fix the SQL
            print('Asking LLM to fix the SQL...')
            correction_prompt = build_correction_prompt(user_question, sql_query, str(error), schema_string)
            sql_query = get_sql_from_llm(correction_prompt)
            print(f'Corrected SQL: {sql_query}')

print('Self-correction loop defined successfully!')

Self-correction loop defined successfully!


In [9]:
# -----------------------------------------------------------
# CELL 9: Test the Self-Correction Loop
# We intentionally pass a broken SQL query to test that
# the self-correction loop catches the error and fixes it.
# -----------------------------------------------------------

# Test with a deliberately broken SQL query
broken_sql = "SELECT * FROM NonExistentTable LIMIT 5"
test_question = "Which artist has the most albums?"

print('Testing self-correction loop with a broken SQL query...')
print('---')
results = run_sql_with_correction(test_question, broken_sql)
print(f'Final results: {results}')

Testing self-correction loop with a broken SQL query...
---
Attempt 1 failed: no such table: NonExistentTable
Asking LLM to fix the SQL...
Corrected SQL: SELECT ArtistId, COUNT(AlbumId) AS AlbumCount 
FROM Album 
GROUP BY ArtistId 
ORDER BY AlbumCount DESC 
LIMIT 1;
SQL succeeded on attempt 2
Final results: [(90, 21)]


In [10]:
# -----------------------------------------------------------
# CELL 10: Updated Full Pipeline with Self-Correction
# We replace run_sql() with run_sql_with_correction()
# so the full pipeline now automatically handles SQL errors.
# -----------------------------------------------------------

def text_to_sql_pipeline_v2(user_question):
    print(f'Question: {user_question}')
    print('---')

    # Step 1: Build the prompt
    prompt = build_prompt(schema_string, user_question)

    # Step 2: Get SQL from LLM
    sql_query = get_sql_from_llm(prompt)
    print(f'Generated SQL: {sql_query}')
    print('---')

    # Step 3: Run SQL with self-correction
    results = run_sql_with_correction(user_question, sql_query)
    print(f'Results: {results}')
    return results

# Test the updated pipeline
text_to_sql_pipeline_v2("Which artist has the most albums?")
print()
text_to_sql_pipeline_v2("What are the top 5 genres by number of tracks?")
print()
text_to_sql_pipeline_v2("How many customers are from the USA?")

Question: Which artist has the most albums?
---
Generated SQL: SELECT Artist.Name, COUNT(Album.AlbumId) AS AlbumCount
FROM Artist
JOIN Album ON Artist.ArtistId = Album.ArtistId
GROUP BY Artist.ArtistId
ORDER BY AlbumCount DESC
LIMIT 1;
---
Results: [('Iron Maiden', 21)]

Question: What are the top 5 genres by number of tracks?
---
Generated SQL: SELECT g.Name, COUNT(t.TrackId) AS TrackCount
FROM Genre g
JOIN Track t ON g.GenreId = t.GenreId
GROUP BY g.GenreId
ORDER BY TrackCount DESC
LIMIT 5;
---
Results: [('Rock', 1297), ('Latin', 579), ('Metal', 374), ('Alternative & Punk', 332), ('Jazz', 130)]

Question: How many customers are from the USA?
---
Generated SQL: SELECT COUNT(*) FROM Customer WHERE Country = 'USA';
---
Results: [(13,)]


[(13,)]